# PyTorch TrainJob from Feast DataFrame

This notebook is for **cluster training from Jupyter** using Kubeflow Trainer.

Flow:
1. Set namespace and artifact inputs.
2. Define a self-contained training function.
3. Submit a distributed `TrainJob`.
4. Monitor status/logs and confirm completion.

It consumes artifacts generated by `feast_data_prep.ipynb`.

## 1) Install dependencies and set target namespace

If needed, install Kubeflow Trainer SDK and training dependencies.

Set `TARGET_NAMESPACE` to the namespace where the TrainJob must be created.

In [ ]:
# !pip install -U kubeflow
# !pip install torch pandas scikit-learn pyarrow

TARGET_NAMESPACE = "kapil-test-namespace"  # <- user can change this
if not TARGET_NAMESPACE:
    raise ValueError("TARGET_NAMESPACE must be set before submitting TrainJob")
print(f"Target namespace: {TARGET_NAMESPACE}")

## 2) Define self-contained training function

Like upstream Trainer examples, all imports are inside the function so it can be serialized and executed in isolated workers.

In [ ]:
def train_fraud_from_feast(
    num_epochs=10,
    batch_size=512,
    lr=1e-3,
    hidden_dim=64,
    training_data_path="/mnt/data/feast_training.parquet",
    metadata_path="/mnt/data/feast_training_metadata.json",
    output_dir="/mnt/models",
):
    import json
    import os
    import random

    import numpy as np
    import pandas as pd
    import torch
    import torch.distributed as dist
    from sklearn.metrics import roc_auc_score
    from torch import nn
    from torch.utils.data import DataLoader, Dataset, DistributedSampler

    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)

    class TabularDataset(Dataset):
        def __init__(self, frame, feature_cols, label_col):
            self.x = torch.tensor(frame[feature_cols].values, dtype=torch.float32)
            self.y = torch.tensor(frame[label_col].values, dtype=torch.float32).unsqueeze(1)

        def __len__(self):
            return len(self.x)

        def __getitem__(self, idx):
            return self.x[idx], self.y[idx]

    class FraudMLP(nn.Module):
        def __init__(self, input_dim, hidden):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(hidden, hidden // 2),
                nn.ReLU(),
                nn.Linear(hidden // 2, 1),
            )

        def forward(self, x):
            return self.net(x)

    if not os.path.exists(training_data_path):
        raise FileNotFoundError(
            f"Missing training dataset at {training_data_path}. Run feast_data_prep.ipynb first."
        )

    with open(metadata_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    feature_cols = metadata["feature_columns"]
    label_col = metadata["label_column"]

    df = pd.read_parquet(training_data_path)
    train_df = df[df["split"] == "train"].copy()
    val_df = df[df["split"] == "val"].copy()

    if train_df.empty or val_df.empty:
        raise RuntimeError("Train/val splits are empty. Rebuild data artifact from feast_data_prep.ipynb.")

    world_size = int(os.getenv("WORLD_SIZE", "1"))
    rank = int(os.getenv("RANK", "0"))
    local_rank = int(os.getenv("LOCAL_RANK", "0"))

    distributed = world_size > 1
    if distributed:
        backend = "nccl" if torch.cuda.is_available() else "gloo"
        dist.init_process_group(backend=backend)

    if torch.cuda.is_available():
        device = torch.device(f"cuda:{local_rank}")
        torch.cuda.set_device(device)
    else:
        device = torch.device("cpu")

    model = FraudMLP(input_dim=len(feature_cols), hidden=hidden_dim).to(device)
    if distributed:
        model = nn.parallel.DistributedDataParallel(
            model,
            device_ids=[local_rank] if torch.cuda.is_available() else None,
        )

    train_dataset = TabularDataset(train_df, feature_cols, label_col)
    val_dataset = TabularDataset(val_df, feature_cols, label_col)

    train_sampler = DistributedSampler(train_dataset) if distributed else None
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
        shuffle=(train_sampler is None),
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)

        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        epoch_loss = running_loss / max(len(train_loader), 1)

        model.eval()
        val_targets = []
        val_scores = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                logits = model(xb)
                probs = torch.sigmoid(logits).cpu().numpy().reshape(-1)
                val_scores.extend(probs.tolist())
                val_targets.extend(yb.numpy().reshape(-1).tolist())

        # Guard against rare single-class validation slices.
        if len(set(int(v) for v in val_targets)) >= 2:
            val_auc = roc_auc_score(val_targets, val_scores)
        else:
            val_auc = float("nan")

        if rank == 0:
            print(
                f"Epoch {epoch + 1}/{num_epochs} | loss={epoch_loss:.4f} | val_auc={val_auc:.4f}"
            )

    if rank == 0:
        os.makedirs(output_dir, exist_ok=True)
        model_to_save = model.module if hasattr(model, "module") else model
        model_path = os.path.join(output_dir, "fraud_mlp_state_dict.pt")
        metrics_path = os.path.join(output_dir, "metrics.json")

        torch.save(
            {
                "model_state_dict": model_to_save.state_dict(),
                "feature_columns": feature_cols,
                "label_column": label_col,
                "hidden_dim": hidden_dim,
            },
            model_path,
        )

        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "val_auc": float(val_auc),
                    "epochs": int(num_epochs),
                    "train_rows": int(len(train_df)),
                    "val_rows": int(len(val_df)),
                },
                f,
                indent=2,
            )

        print(f"Saved model artifact: {model_path}")
        print(f"Saved metrics artifact: {metrics_path}")

    if distributed:
        dist.barrier()
        dist.destroy_process_group()

## 3) Check runtime availability in the cluster

Why run this: confirms the expected runtime (for example `torch-distributed`) is installed before submitting a TrainJob.

In [ ]:
from kubeflow.trainer import TrainerClient

client = TrainerClient()
runtimes = list(client.list_runtimes())
for rt in runtimes:
    print(rt)

if not any(rt.name == "torch-distributed" for rt in runtimes):
    raise RuntimeError("Runtime 'torch-distributed' not found in this cluster")

## 4) Submit distributed TrainJob to Kubernetes

Why run this: creates the actual Trainer `TrainJob` in your target namespace using the Feast-prepared artifacts.

Tune `num_nodes` and `resources_per_node` for your cluster capacity.

In [ ]:
from kubeflow.trainer import CustomTrainer, TrainerClient

if not TARGET_NAMESPACE:
    raise ValueError("TARGET_NAMESPACE must be set")

# Prefer namespace-aware client construction if available.
train_kwargs = {}
job_scope_kwargs = {}
try:
    client = TrainerClient(namespace=TARGET_NAMESPACE)
except TypeError:
    client = TrainerClient()
    train_kwargs["namespace"] = TARGET_NAMESPACE
    job_scope_kwargs["namespace"] = TARGET_NAMESPACE

trainer = CustomTrainer(
    func=train_fraud_from_feast,
    func_args={
        "num_epochs": 10,
        "batch_size": 512,
        "lr": 1e-3,
        "hidden_dim": 64,
        "training_data_path": "/mnt/data/feast_training.parquet",
        "metadata_path": "/mnt/data/feast_training_metadata.json",
        "output_dir": "/mnt/models",
    },
    num_nodes=2,
    resources_per_node={
        "cpu": "2",
        "memory": "4Gi",
        # "nvidia.com/gpu": 1,
    },
    packages_to_install=["torch", "pandas", "scikit-learn", "pyarrow"],
)

job_name = client.train(
    trainer=trainer,
    runtime="torch-distributed",
    **train_kwargs,
)

print(f"Submitted TrainJob: {job_name} in namespace: {TARGET_NAMESPACE}")

## 5) Monitor TrainJob and logs

Why run this: verifies the job starts correctly and lets attendees watch progress/errors directly from notebook output.

In [ ]:
client.wait_for_job_status(name=job_name, status={"Running"}, timeout=300, **job_scope_kwargs)
print(f"{job_name} is running in namespace {TARGET_NAMESPACE}. Streaming logs:")
for line in client.get_job_logs(job_name, follow=True, **job_scope_kwargs):
    print(line, end="")

In [ ]:
client.wait_for_job_status(name=job_name, timeout=20, **job_scope_kwargs)
print(f"TrainJob completed in namespace {TARGET_NAMESPACE}.")